# NeuraVol Forecast v2: factor neural model with shrinkage

What changed from v1 (v1 did not beat "no change" overall; see `reports/nn_full/`):

1. **Cleaner training targets**: one-day closing-price spikes in the surface history are filtered before training.
2. **Fewer numbers to predict**: the model forecasts moves in 8 surface factors (level, skew, term slope...) instead of 63 cells.
3. **One model for all horizons**: 1, 5 and 20 days are learned together, which regularises the long horizon.
4. **Event calendar features**: Union Budget, general-election results, month-end expiry week, plus optional RBI dates.
5. **Huber loss** so crash days do not dominate, and **shrinkage**: final forecast = no change + α × (model's predicted move),
   with α chosen on validation data, so the model only moves the forecast as far as it has earned.

Scoring is identical to v1: walk-forward 2020-2026, errors on the **raw** surfaces, cells backed by real quotes only.

**Run:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`. Upload `NIFTY_grid.npz` and
`NIFTY_diagnostics.csv.gz` (both from `data/surfaces/`; select both in one dialog). Optional: `events.csv` (see section 1).

In [ ]:
QUICK = False            # True = ~3 min smoke test (one test year, one seed, few epochs)
USE_DRIVE = False
DRIVE_DIR = "/content/drive/MyDrive/NEURAVOLAFORECAST"

HORIZONS = [1, 5, 20]
LOOKBACK = 20
N_FACTORS = 8
TEST_YEARS = list(range(2020, 2027)) if not QUICK else [2024]
SEEDS = [0, 1, 2, 3, 4] if not QUICK else [0]
MAX_EPOCHS = 200 if not QUICK else 15
PATIENCE = 25
COVID = ("2020-02-15", "2020-06-30")
SPIKE_THRESHOLD = 0.12   # a cell is a spike if it jumps >12% (log) vs BOTH neighbours in the same direction

In [ ]:
import os, json, time, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

NEEDED = ["NIFTY_grid.npz", "NIFTY_diagnostics.csv.gz"]
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = DRIVE_DIR
elif IN_COLAB:
    from google.colab import files
    DATA_DIR = "/content"
    while True:
        missing = [f for f in NEEDED if not os.path.exists(os.path.join(DATA_DIR, f))]
        if not missing:
            break
        print("Please upload (select several files in one dialog):", ", ".join(missing), "(+ optional events.csv)")
        files.upload()
else:
    DATA_DIR = os.environ.get("NEURAVOL_DATA", ".")
for f in NEEDED + ["events.csv"]:
    p = os.path.join(DATA_DIR, f)
    print(f"{'found' if os.path.exists(p) else ('MISSING' if f in NEEDED else 'not provided (optional)')}: {f}")
OUT_DIR = os.path.join("/content" if IN_COLAB else DATA_DIR, "nn_v2_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

## 1. Data, cleaning and features

**Events.** Built in: Union Budget days and general-election result days (both scheduled in advance, so no look-ahead).
To add RBI policy dates or anything else, upload an `events.csv` with columns `date,event`
(e.g. `2024-06-07,RBI`). Every distinct `event` name becomes its own pair of features.

In [ ]:
z = np.load(os.path.join(DATA_DIR, "NIFTY_grid.npz"))
dates = pd.to_datetime(z["dates"])
IV = z["iv"].astype(np.float64)
MASK = z["mask"].astype(bool)
TENORS = z["tenors"].astype(float)
MON = z["moneyness"].astype(float)
D, NT, NK = IV.shape
NC = NT * NK
TAU = np.repeat(TENORS / 365.0, NK)
IVF = IV.reshape(D, NC)
MASKF = MASK.reshape(D, NC)
LOGIV = np.log(IVF)

# --- spike filter (used for TRAINING TARGETS only; inputs and scoring stay raw) ---
def despike(x, thr):
    x = x.copy()
    d_prev = x[1:-1] - x[:-2]
    d_next = x[1:-1] - x[2:]
    spike = (np.sign(d_prev) == np.sign(d_next)) & (np.minimum(np.abs(d_prev), np.abs(d_next)) > thr)
    fix = 0.5 * (x[:-2] + x[2:])
    x[1:-1] = np.where(spike, fix, x[1:-1])
    return x, spike


LOGIV_CLEAN, spikes = despike(LOGIV, SPIKE_THRESHOLD)
print(f"spike filter: {spikes.sum()} of {spikes.size} cell-days replaced ({spikes.mean():.2%}), "
      f"on {spikes.any(1).sum()} days")

# --- market features from the near forward ---
diag = pd.read_csv(os.path.join(DATA_DIR, "NIFTY_diagnostics.csv.gz"), parse_dates=["date"]).set_index("date").reindex(dates)
fwd = diag["F_near"].ffill().values
ret = np.clip(np.r_[0.0, np.diff(np.log(fwd))], -0.15, 0.15)
rs = pd.Series(ret)
MKT = np.c_[ret * 100, np.abs(ret) * 100,
            rs.rolling(5, min_periods=1).std().fillna(0).values * np.sqrt(252) * 100,
            rs.rolling(21, min_periods=1).std().fillna(0).values * np.sqrt(252) * 100]

# --- event calendar features ---
BUDGET = ["2016-02-29", "2017-02-01", "2018-02-01", "2019-02-01", "2019-07-05", "2020-02-01", "2021-02-01",
          "2022-02-01", "2023-02-01", "2024-02-01", "2024-07-23", "2025-02-01", "2026-02-01"]
ELECTION_RESULTS = ["2019-05-23", "2024-06-04"]
events = {"budget": pd.to_datetime(BUDGET), "election": pd.to_datetime(ELECTION_RESULTS)}
ev_path = os.path.join(DATA_DIR, "events.csv")
if os.path.exists(ev_path):
    ev = pd.read_csv(ev_path, parse_dates=["date"])
    for name, g in ev.groupby("event"):
        events[str(name).lower()] = pd.DatetimeIndex(g["date"].sort_values())


def event_features(dts, ev_dates, ahead_scale=10.0, after_scale=5.0):
    """Two features per event type: closeness of the NEXT event (calendar days) and of the LAST one."""
    ev_dates = pd.DatetimeIndex(sorted(ev_dates))
    nxt = np.full(len(dts), np.inf)
    prv = np.full(len(dts), np.inf)
    for i, d in enumerate(dts):
        j = ev_dates.searchsorted(d)                    # first event >= d
        if j < len(ev_dates):
            nxt[i] = (ev_dates[j] - d).days
        if j > 0:
            prv[i] = (d - ev_dates[j - 1]).days
    return np.c_[np.exp(-nxt / ahead_scale), np.exp(-prv / after_scale)]


EVT = [event_features(dates, v) for v in events.values()]
# month-end / monthly-expiry week and day of week
month_end = dates + pd.offsets.BMonthEnd(0)
EVT.append(np.c_[np.exp(-(month_end - dates).days.values / 5.0)])
EVT.append(np.eye(5)[np.clip(dates.dayofweek.values, 0, 4)])
EVT = np.concatenate(EVT, axis=1)
EXO = np.c_[MKT, EVT].astype(np.float64)
NE = EXO.shape[1]
print(f"{D} days {dates[0].date()} -> {dates[-1].date()} | event types: {list(events)} | exogenous features: {NE}")

## 2. Scoring helpers and baselines (identical to v1)

In [ ]:
def rmse_vp(pred_iv, true_iv, mask):
    e = (pred_iv - true_iv)[mask] * 100
    return float(np.sqrt(np.mean(e ** 2))) if e.size else float("nan")


def daily_loss(pred_iv, true_iv, mask):
    se = ((pred_iv - true_iv) * 100) ** 2
    return np.array([se[i][mask[i]].mean() if mask[i].any() else np.nan for i in range(len(se))])


def diebold_mariano(loss_a, loss_b, h):
    d = np.asarray(loss_a) - np.asarray(loss_b)
    d = d[~np.isnan(d)]
    n = len(d)
    if n < 30:
        return float("nan"), float("nan")
    var = np.var(d)
    for l in range(1, max(h - 1, 0) + 1):
        var += 2 * (1 - l / h) * np.cov(d[l:], d[:-l], ddof=0)[0, 1]
    stat = d.mean() / np.sqrt(var / n)
    return float(stat), float(norm.cdf(stat))


def region_table(pred_iv, true_iv, mask):
    P, T_, M = (a.reshape(-1, NT, NK) for a in (pred_iv, true_iv, mask))
    out = {"overall": rmse_vp(pred_iv, true_iv, mask)}
    for i, t in enumerate(TENORS):
        out[f"{int(t)}d"] = rmse_vp(P[:, i], T_[:, i], M[:, i])
    for name, sel in {"put wing": MON <= -0.05, "near ATM": np.abs(MON) <= 0.025, "call wing": MON >= 0.05}.items():
        out[name] = rmse_vp(P[:, :, sel], T_[:, :, sel], M[:, :, sel])
    return out


def fit_pca_var(X, k=3):
    mu = X.mean(0)
    _, _, Vt = np.linalg.svd(X - mu, full_matrices=False)
    V = Vt[:k].T
    F = (X - mu) @ V
    B = np.linalg.lstsq(np.c_[np.ones(len(F) - 1), F[:-1]], F[1:], rcond=None)[0]
    return mu, V, B


def pca_var_forecast(x, model, h):
    mu, V, B = model
    f = (x - mu) @ V
    for _ in range(h):
        f = B[0] + f @ B[1:]
    return mu + f @ V.T


HMAX = max(HORIZONS)


def test_indices(year):
    """Forecast origins in `year` (same set for every horizon: all targets up to HMAX must exist)."""
    return np.where((dates.year == year) & (np.arange(D) >= LOOKBACK - 1) & (np.arange(D) + HMAX < D))[0]


def baselines(h, origins):
    rw = IVF[origins]
    pv = np.empty_like(rw)
    model, last_fit = None, -10 ** 9
    for j, t in enumerate(origins):
        if t - last_fit >= 21:
            model, last_fit = fit_pca_var(IVF[: t + 1]), t
        pv[j] = pca_var_forecast(IVF[t], model, h)
    return {"No change": rw, "PCA-VAR": pv}

## 3. Factor space
PCA of log-IV fitted on the training window only. The model predicts the change in factor scores;
the forecast is today's surface plus that change mapped back to the grid, so "no move" = "no change" exactly.

In [ ]:
def fit_factors(train_rows, k=N_FACTORS):
    """PCA of log-IV, with extrapolated cells down-weighted so factors describe quoted parts of the surface."""
    X = LOGIV_CLEAN[train_rows]
    W = np.where(MASKF[train_rows], 1.0, 0.1)
    mu = X.mean(0)
    _, S, Vt = np.linalg.svd((X - mu) * W, full_matrices=False)
    return dict(mu=mu, V=Vt[:k].T, explained=float((S[:k] ** 2).sum() / (S ** 2).sum()))


def factor_scores(fac, logiv):
    return (logiv - fac["mu"]) @ fac["V"]

In [ ]:
# quick check of the data layer
_fac = fit_factors(np.arange(0, 1000))
print(f"{N_FACTORS} factors explain {_fac['explained']:.1%} of surface variance (2016-2019 window)")
_o = test_indices(2024)
for h in HORIZONS:
    b = baselines(h, _o)
    print(f"2024 h={h:2d}: " + ", ".join(f"{k} {rmse_vp(v, IVF[_o + h], MASKF[_o + h]):.2f}" for k, v in b.items()))

## 4. The v2 model
GRU over the last 20 days of [factor scores, market features, event features] -> predicted change of each
factor at 1, 5 and 20 days (one model, three outputs). The output layer starts at zero (= "no change").

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as Fn

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEV, torch.cuda.get_device_name(0) if DEV.type == "cuda" else "")
H = len(HORIZONS)
HZ = np.array(HORIZONS)
LAMBDA_ARB = 1.0
TAU_T = torch.tensor(TAU, dtype=torch.float32, device=DEV)


class FactorNet(nn.Module):
    def __init__(self, k=N_FACTORS, n_exo=NE, hid=64, drop=0.25):
        super().__init__()
        self.k = k
        self.enc = nn.Sequential(nn.Linear(k + n_exo, hid), nn.GELU(), nn.Dropout(drop), nn.Linear(hid, hid), nn.GELU())
        self.gru = nn.GRU(hid, hid, batch_first=True)
        self.head = nn.Sequential(nn.Linear(hid + k, hid), nn.GELU(), nn.Dropout(drop), nn.Linear(hid, H * k))
        nn.init.zeros_(self.head[-1].weight)
        nn.init.zeros_(self.head[-1].bias)

    def forward(self, seq, last):
        h, _ = self.gru(self.enc(seq))
        return self.head(torch.cat([h[:, -1], last], dim=1)).view(-1, H, self.k)


def make_prep(train_idx):
    """Everything fitted on the training window only: factors, input scaling, output scaling, loss scales."""
    rows = np.arange(0, train_idx.max() + 1)
    fac = fit_factors(rows)
    F_all = factor_scores(fac, LOGIV)
    feats = np.c_[F_all, EXO]
    lag_rows = np.unique(np.concatenate([train_idx - j for j in range(LOOKBACK)]))
    mu, sd = feats[lag_rows].mean(0), feats[lag_rows].std(0) + 1e-6
    dF = np.stack([factor_scores(fac, LOGIV_CLEAN[train_idx + h]) - F_all[train_idx] for h in HORIZONS], 1)
    ysd = dF.std(0) + 1e-4                                           # (H, k)
    scale = np.array([rmse_vp(IVF[train_idx], np.exp(LOGIV_CLEAN[train_idx + h]), MASKF[train_idx + h])
                      for h in HORIZONS])                            # typical no-change error per horizon
    return dict(fac=fac, S=(feats - mu) / sd, ysd=ysd, scale=scale)


def tensors(idx, prep):
    t = lambda a: torch.tensor(np.asarray(a), dtype=torch.float32, device=DEV)
    S = prep["S"]
    b = dict(seq=t(S[idx[:, None] - np.arange(LOOKBACK - 1, -1, -1)[None, :]]),
             last=t(S[idx, :N_FACTORS]), logiv=t(LOGIV[idx]))
    if (idx + HMAX < D).all():
        tgt_rows = idx[:, None] + HZ[None, :]                         # (B, H)
        b["clean"] = t(np.exp(LOGIV_CLEAN[tgt_rows]))                 # training target
        b["raw"] = t(IVF[tgt_rows])                                   # scoring target
        b["w"] = t(np.where(MASKF[tgt_rows], 1.0, 0.1))
        b["m"] = t(MASKF[tgt_rows].astype(np.float32))
    return b


class Consts:
    def __init__(self, prep):
        t = lambda a: torch.tensor(np.asarray(a), dtype=torch.float32, device=DEV)
        self.V, self.ysd, self.scale = t(prep["fac"]["V"]), t(prep["ysd"]), t(prep["scale"])


def pred_dlog(model, b, c):
    """Predicted change in log-IV on the grid: (B, H, NC)."""
    dF = model(b["seq"], b["last"]) * c.ysd
    return torch.einsum("bhk,ck->bhc", dF, c.V)


def loss_fn(model, b, c):
    iv = torch.exp(b["logiv"][:, None, :] + pred_dlog(model, b, c))
    err = (iv - b["clean"]) * 100 / c.scale[None, :, None]
    hub = Fn.smooth_l1_loss(err, torch.zeros_like(err), beta=1.0, reduction="none")
    loss = (b["w"] * hub).sum() / b["w"].sum()
    wt = (iv ** 2 * TAU_T).view(-1, H, NT, NK)
    return loss + LAMBDA_ARB * 1e4 * torch.relu(wt[:, :, :-1] - wt[:, :, 1:]).sum(dim=(2, 3)).mean()


def rel_rmse(dlog, b):
    """Mean over horizons of (model RMSE / no-change RMSE) on raw targets, quoted cells only."""
    iv = torch.exp(b["logiv"][:, None, :] + dlog)
    rw = torch.exp(b["logiv"])[:, None, :].expand_as(iv)
    r = []
    for j in range(H):
        m = b["m"][:, j]
        e1 = (((iv[:, j] - b["raw"][:, j]) * 100) ** 2 * m).sum() / m.sum()
        e0 = (((rw[:, j] - b["raw"][:, j]) * 100) ** 2 * m).sum() / m.sum()
        r.append(torch.sqrt(e1 / e0))
    return torch.stack(r)


def subset(b, i):
    return {k: v[i] for k, v in b.items()}


def fit_one(train_idx, val_idx, prep, seed, bs=64, lr=1e-3):
    torch.manual_seed(seed); np.random.seed(seed)
    c = Consts(prep)
    model = FactorNet().to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    tr, va = tensors(train_idx, prep), tensors(val_idx, prep)
    best, best_state, wait, n = float("inf"), None, 0, len(train_idx)
    for ep in range(MAX_EPOCHS):
        model.train()
        perm = torch.randperm(n, device=DEV)
        for s in range(0, n, bs):
            loss = loss_fn(model, subset(tr, perm[s:s + bs]), c)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        model.eval()
        with torch.no_grad():
            vl = rel_rmse(pred_dlog(model, va, c), va).mean().item()
        if vl < best - 1e-5:
            best, wait = vl, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    model.load_state_dict(best_state)
    model.eval()
    return model, ep + 1, best


def ens_dlog(models, idx, prep):
    c, b = Consts(prep), tensors(idx, prep)
    with torch.no_grad():
        return torch.stack([pred_dlog(m, b, c) for m in models]).mean(0), b


def choose_alpha(models, val_idx, prep):
    """Shrinkage per horizon: forecast = today + alpha * predicted move, alpha picked on validation."""
    dl, b = ens_dlog(models, val_idx, prep)
    grid = np.round(np.arange(0, 1.21, 0.05), 2)
    alphas = []
    for j in range(H):
        scores = [rel_rmse(dl * a, b)[j].item() for a in grid]
        alphas.append(float(grid[int(np.argmin(scores))]))
    return np.array(alphas)


def split_train_val(first_test, val_frac=0.15):
    usable = np.arange(LOOKBACK - 1, first_test - HMAX)
    n_val = max(60, int(len(usable) * val_frac))
    val = usable[-n_val:]
    return usable[usable + HMAX < val[0]], val           # embargo: training targets end before validation

## 5. Walk-forward evaluation 2020-2026 (retrain every year, 5-seed ensemble, shrinkage)

In [ ]:
t0 = time.time()
origins = np.concatenate([test_indices(y) for y in TEST_YEARS])
raw_pred = np.empty((len(origins), H, NC))
shr_pred = np.empty_like(raw_pred)
alpha_log = {}
for y in TEST_YEARS:
    test = test_indices(y)
    train, val = split_train_val(test[0])
    prep = make_prep(train)
    fits = [fit_one(train, val, prep, s) for s in SEEDS]
    models = [f[0] for f in fits]
    alpha = choose_alpha(models, val, prep)
    alpha_log[y] = alpha.tolist()
    dl, _ = ens_dlog(models, test, prep)
    dl = dl.cpu().numpy()
    sel = np.isin(origins, test)
    raw_pred[sel] = np.exp(LOGIV[test][:, None, :] + dl)
    shr_pred[sel] = np.exp(LOGIV[test][:, None, :] + dl * alpha[None, :, None])
    print(f"{y}: train {len(train)} val {len(val)} test {len(test)} | factors explain {prep['fac']['explained']:.1%} | "
          f"epochs {[f[1] for f in fits]} | val rel-RMSE {np.mean([f[2] for f in fits]):.3f} | "
          f"alpha {dict(zip(HORIZONS, alpha))} | {time.time() - t0:.0f}s")

## 6. Results

In [ ]:
results, yearly, store = {}, {}, {"origins": origins}
calm = ~((dates[origins] >= COVID[0]) & (dates[origins] <= COVID[1]))
for j, h in enumerate(HORIZONS):
    tgt, msk = IVF[origins + h], MASKF[origins + h]
    preds = {**baselines(h, origins), "NN v2 (raw)": raw_pred[:, j], "NN v2 (shrunk)": shr_pred[:, j]}
    preds["NN v2 shrunk + PCA-VAR avg"] = 0.5 * (preds["NN v2 (shrunk)"] + preds["PCA-VAR"])
    tab = pd.DataFrame({k: region_table(v, tgt, msk) for k, v in preds.items()})
    tab.loc["excl. COVID crash"] = [rmse_vp(v[calm], tgt[calm], msk[calm]) for v in preds.values()]
    clean_tgt = np.exp(LOGIV_CLEAN[origins + h])
    tab.loc["vs spike-filtered target"] = [rmse_vp(v, clean_tgt, msk) for v in preds.values()]
    losses = {k: daily_loss(v, tgt, msk) for k, v in preds.items()}
    dm = {f"{a} vs {b_}": diebold_mariano(losses[a], losses[b_], h)
          for a in ["NN v2 (shrunk)", "NN v2 shrunk + PCA-VAR avg"] for b_ in ["No change", "PCA-VAR"]}
    yr = dates[origins].year
    yearly[h] = pd.DataFrame({y: {k: rmse_vp(v[yr == y], tgt[yr == y], msk[yr == y]) for k, v in preds.items()}
                              for y in TEST_YEARS}).T
    arb = float((((preds["NN v2 (shrunk)"] ** 2 * TAU).reshape(-1, NT, NK)[:, :-1] -
                  (preds["NN v2 (shrunk)"] ** 2 * TAU).reshape(-1, NT, NK)[:, 1:]) > 1e-8).mean())
    results[h] = dict(table=tab, dm=dm, arb=arb)
    for k, v in preds.items():
        store[f"h{h}_{k}"] = v
    store[f"h{h}_target"], store[f"h{h}_mask"] = tgt, msk
    print(f"\n=== {h}-day horizon: RMSE in vol points, {len(origins)} forecasts ===")
    print(tab.round(2).to_string())
    print("\nby year:\n" + yearly[h].round(2).to_string())
    for k, (s_, p_) in dm.items():
        print(f"DM {k}: stat {s_:.2f}, p = {p_:.3f} {'<- significantly better' if p_ < 0.05 else ''}")
    print(f"calendar-arbitrage violations (NN v2 shrunk): {arb:.2%}")

In [ ]:
overall = pd.DataFrame({f"{h}d": r["table"].loc["overall"] for h, r in results.items()}).T
calm_tab = pd.DataFrame({f"{h}d": r["table"].loc["excl. COVID crash"] for h, r in results.items()}).T
print("Overall\n", overall.round(2), "\n\nExcluding Feb-Jun 2020\n", calm_tab.round(2))
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
overall.plot.bar(ax=ax[0], rot=0, title="Forecast error, all test days")
calm_tab.plot.bar(ax=ax[1], rot=0, title="Forecast error, excluding COVID crash")
for a in ax:
    a.set_ylabel("RMSE, vol points"); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "v2_rmse_by_horizon.png"), dpi=130); plt.show()

c30 = int(np.where(TENORS == 30)[0][0]) * NK + int(np.argmin(np.abs(MON)))
fig, ax = plt.subplots(len(HORIZONS), 1, figsize=(14, 3.2 * len(HORIZONS)), sharex=True)
for a, (j, h) in zip(np.atleast_1d(ax), enumerate(HORIZONS)):
    d = dates[origins + h]
    a.plot(d, IVF[origins + h, c30] * 100, lw=1, label="actual")
    a.plot(d, shr_pred[:, j, c30] * 100, lw=1, label=f"NN v2 forecast made {h}d earlier")
    a.plot(d, IVF[origins, c30] * 100, lw=0.7, alpha=0.5, label="no change")
    a.set_ylabel("30d ATM IV %"); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "v2_atm30_forecasts.png"), dpi=130); plt.show()

with open(os.path.join(OUT_DIR, "nn_v2_results.json"), "w") as f:
    json.dump({"alphas_by_year": alpha_log, "spike_cells_replaced": int(spikes.sum()),
               "by_horizon": {str(h): dict(table=r["table"].round(4).to_dict(), dm=r["dm"], calendar_violation_rate=r["arb"],
                                           by_year=yearly[h].round(4).to_dict()) for h, r in results.items()}}, f, indent=2)
np.savez_compressed(os.path.join(OUT_DIR, "nn_v2_test_predictions.npz"), **{k.replace(" ", "_"): v for k, v in store.items()})

## 7. Final model on all data + forecast from the latest day
Saved as `neuravol_v2.pt`: everything the dashboard needs (factors, scaling, alphas, weights).

In [ ]:
train, val = split_train_val(D, val_frac=0.10)
prep = make_prep(train)
models = [fit_one(train, val, prep, s)[0] for s in SEEDS]
alpha = choose_alpha(models, val, prep)
latest = np.array([D - 1])
dl, _ = ens_dlog(models, latest, prep)
fc = np.exp(LOGIV[latest][:, None, :] + dl.cpu().numpy() * alpha[None, :, None])[0]     # (H, NC)
rows = []
for j, h in enumerate(HORIZONS):
    for i, tn in enumerate(TENORS):
        for kk, m in enumerate(MON):
            c = i * NK + kk
            rows.append(dict(as_of=dates[-1].date(), horizon_days=h, tenor_days=int(tn), moneyness=m,
                             iv_today=IVF[-1, c], iv_forecast=fc[j, c], backed_by_quotes=bool(MASKF[-1, c])))
fc_df = pd.DataFrame(rows)
fc_df.to_csv(os.path.join(OUT_DIR, "latest_forecast_v2.csv"), index=False)
torch.save({"version": 2, "horizons": HORIZONS, "lookback": LOOKBACK, "n_factors": N_FACTORS,
            "tenors": TENORS.tolist(), "moneyness": MON.tolist(), "last_date": str(dates[-1].date()),
            "events": {k: [str(d.date()) for d in v] for k, v in events.items()},
            "factors": {"mu": prep["fac"]["mu"].tolist(), "V": prep["fac"]["V"].tolist()},
            "ysd": prep["ysd"].tolist(), "scale": prep["scale"].tolist(), "alpha": alpha.tolist(),
            "state_dicts": [{k: v.cpu() for k, v in m.state_dict().items()} for m in models]},
           os.path.join(OUT_DIR, "neuravol_v2.pt"))
print("final alphas:", dict(zip(HORIZONS, alpha)))
print(fc_df[fc_df.moneyness == 0].round(4).to_string(index=False))

In [ ]:
if IN_COLAB:
    import shutil
    from google.colab import files
    shutil.make_archive("/content/nn_v2_outputs", "zip", OUT_DIR)
    files.download("/content/nn_v2_outputs.zip")
    print("Downloaded nn_v2_outputs.zip: put it in the NEURAVOLAFORECAST folder")